In [ ]:
import urllib.request
import gzip

url = "ftp://ita.ee.lbl.gov/traces/calgary_access_log.gz"
file_path = "calgary_access_log.gz"

urllib.request.urlretrieve(url, file_path)

with gzip.open(file_path, 'rt', encoding='latin1') as f:
    lines = f.readlines()


In [ ]:
log_line = 'local - - [24/Oct/1994:14:23:15 -0600] "GET /1234.html HTTP/1.0" 200 2048'



In [ ]:
import re
from datetime import datetime



In [ ]:
pattern = re.compile(
    r'(\S+) - - \[(.*?)\] "(?:GET|POST) (.*?) HTTP/\d\.\d" (\d{3}) (\S+)'
)

log_data = []

for line in lines:
    match = pattern.match(line)
    if match:
        host, timestamp_str, file_path, status_code, bytes_sent = match.groups()

        # Convert timestamp
        try:
            timestamp = datetime.strptime(timestamp_str, "%d/%b/%Y:%H:%M:%S %z")
        except ValueError:
            continue  

        # Clean up byte value
        if bytes_sent == "-":
            bytes_sent = None
        else:
            bytes_sent = int(bytes_sent)

        # Extract file name and extension
        filename = file_path.split("/")[-1]
        ext = filename.split(".")[-1] if "." in filename else ""

        log_data.append({
            "host": host,
            "timestamp": timestamp,
            "filename": filename,
            "extension": ext,
            "http_code": int(status_code),
            "bytes": bytes_sent
        })


In [ ]:
import pandas as pd

df = pd.DataFrame(log_data)


Q1: Count of total log records

In [ ]:
len(df)

Q2: Count of unique hosts

In [ ]:
df['host'].nunique()

Q3: Date-wise unique filename counts

In [ ]:
df['date'] = df['timestamp'].dt.strftime('%d-%b-%Y')
unique_files_by_date = df.groupby('date')['filename'].nunique().to_dict()


Q4:Number of 404 response codes

In [ ]:
num_404 = df[df['http_code'] == 404].shape[0]
print("Q4: 404 errors:", num_404)


In [ ]:
Q5: Top 15 filenames with 404 responses

In [ ]:
top_404_files = df[df['http_code'] == 404]['filename'].value_counts().head(15)
print("Q5: Top 15 404 filenames:\n", top_404_files)


In [ ]:
Q6: Top 15 file extensions with 404 responses

In [ ]:
top_404_extensions = df[df['http_code'] == 404]['extension'].value_counts().head(15)
print("Q6: Top 15 404 file extensions:\n", top_404_extensions)


Q7: Total bandwidth transferred per day for July 1995

In [ ]:
july_df = df[
    (df['timestamp'].dt.month == 7) &
    (df['timestamp'].dt.year == 1995) &
    (df['bytes'].notnull())
]

bandwidth_per_day = july_df.groupby(
    july_df['timestamp'].dt.strftime('%d-%b-%Y')
)['bytes'].sum().to_dict()

print("Q7: July 1995 Bandwidth (sample):", dict(list(bandwidth_per_day.items())[:5]))


Q8: Hourly request distribution

In [ ]:
hourly_distribution = df['timestamp'].dt.hour.value_counts().sort_index().to_dict()
print("Q8: Hourly distribution:", hourly_distribution)
